In [1]:
from pathlib import Path

project_path = Path.cwd()
data_path = project_path / "data"
processed_path = data_path / "processed"

print("Project path:", project_path)
print("Processed folder exists:", processed_path.exists())

Project path: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets
Processed folder exists: True


In [3]:
import pandas as pd

exception_csv_path = (
    processed_path / "phase12_exception_records.csv"
)

exceptions_to_load = pd.read_csv(exception_csv_path)

print("Dataset shape:", exceptions_to_load.shape)

print("\nColumn names:")
print(exceptions_to_load.columns.tolist())

print("\nFirst five records:")
display(exceptions_to_load.head())

print("\nMissing values:")
display(exceptions_to_load.isna().sum())

Dataset shape: (685, 12)

Column names:
['exception_id', 'bag_id', 'exception_type', 'exception_description', 'severity', 'priority', 'status', 'assigned_to', 'created_at', 'updated_at', 'resolved_at', 'resolution_notes']

First five records:


,exception_id,bag_id,exception_type,exception_description,severity,priority,status,assigned_to,created_at,updated_at,resolved_at,resolution_notes
0,EXC000001,BAG00501,MISSING_SEGMENT_EVENT,Missing event: SORTED,MEDIUM,2,OPEN,NaN,2026-11-12 05:01:28.929406+00:00,2026-11-12 05:01:28.933841+00:00,NaN,NaN
1,EXC000002,BAG00501,HANDOFF_TIMING_INCONSISTENCY,Second flight loading occurred before the bagg...,HIGH,1,RESOLVED,Investigator_01,2026-11-12 05:01:28.929406+00:00,2026-11-12 05:06:30.699523+00:00,2026-11-12 05:06:30.699523+00:00,Handoff timestamp inconsistency reviewed. Flag...
2,EXC000003,BAG00502,MISSING_SEGMENT_EVENT,Missing event: SORTED,MEDIUM,2,OPEN,NaN,2026-11-12 05:01:28.929406+00:00,2026-11-12 05:01:28.933841+00:00,NaN,NaN
3,EXC000004,BAG00503,MISSING_SEGMENT_EVENT,Missing event: SORTED,MEDIUM,2,OPEN,NaN,2026-11-12 05:01:28.929406+00:00,2026-11-12 05:01:28.933841+00:00,NaN,NaN
4,EXC000005,BAG00504,MISSING_SEGMENT_EVENT,Missing event: SORTED,MEDIUM,2,OPEN,NaN,2026-11-12 05:01:28.929406+00:00,2026-11-12 05:01:28.933841+00:00,NaN,NaN



Missing values:


exception_id               0
bag_id                     0
exception_type             0
exception_description      0
severity                   0
priority                   0
status                     0
assigned_to              684
created_at                 0
updated_at                 0
resolved_at              684
resolution_notes         684
dtype: int64

In [4]:
import importlib.util

print(
    "SQLAlchemy installed:",
    importlib.util.find_spec("sqlalchemy") is not None
)

print(
    "Psycopg2 installed:",
    importlib.util.find_spec("psycopg2") is not None
)

print(
    "Psycopg installed:",
    importlib.util.find_spec("psycopg") is not None
)

SQLAlchemy installed: True
Psycopg2 installed: True
Psycopg installed: False


In [5]:
from sqlalchemy import create_engine, text
from getpass import getpass
from urllib.parse import quote_plus

# Enter your PostgreSQL password locally
db_password = getpass("Enter PostgreSQL password: ")

# Create database connection URL
connection_url = (
    f"postgresql+psycopg2://postgres:"
    f"{quote_plus(db_password)}"
    f"@localhost:5433/airport_baggage_db"
)

# Create SQLAlchemy engine
engine = create_engine(connection_url)

# Test connection
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database(), current_user;")
    )

    print("Database connection successful!")
    print("Database and user:", result.fetchone())

Enter PostgreSQL password:  ········


Database connection successful!
Database and user: ('airport_baggage_db', 'postgres')


In [6]:
# Create a safe copy for PostgreSQL insertion

exceptions_db = exceptions_to_load.copy()

# Convert timestamp columns to UTC-aware datetime
timestamp_columns = [
    "created_at",
    "updated_at",
    "resolved_at"
]

for column in timestamp_columns:
    exceptions_db[column] = pd.to_datetime(
        exceptions_db[column],
        utc=True,
        errors="coerce"
    )

# Convert missing optional values to None
optional_columns = [
    "assigned_to",
    "resolved_at",
    "resolution_notes"
]

for column in optional_columns:
    exceptions_db[column] = exceptions_db[column].where(
        exceptions_db[column].notna(),
        None
    )

print("Prepared dataset shape:", exceptions_db.shape)

print("\nTimestamp data types:")
print(exceptions_db[timestamp_columns].dtypes)

print("\nMissing optional values:")
display(exceptions_db[optional_columns].isna().sum())

Prepared dataset shape: (685, 12)

Timestamp data types:
created_at     datetime64[us, UTC]
updated_at     datetime64[us, UTC]
resolved_at    datetime64[us, UTC]
dtype: object

Missing optional values:


assigned_to         684
resolved_at         684
resolution_notes    684
dtype: int64

In [7]:
from sqlalchemy import text

with engine.connect() as connection:
    existing_count = connection.execute(
        text("SELECT COUNT(*) FROM exception_records")
    ).scalar()

print("Existing exception records:", existing_count)

Existing exception records: 0


In [8]:
# Insert all exception records in one transaction

try:
    with engine.begin() as connection:
        exceptions_db.to_sql(
            name="exception_records",
            con=connection,
            if_exists="append",
            index=False,
            method="multi",
            chunksize=200
        )

    print("Exception records inserted successfully!")

except Exception as error:
    print("Insertion failed.")
    print("Error:", error)

Exception records inserted successfully!


In [9]:
# Verify inserted record count

with engine.connect() as connection:
    db_count = connection.execute(
        text("SELECT COUNT(*) FROM exception_records")
    ).scalar()

print("Python records:", len(exceptions_db))
print("PostgreSQL records:", db_count)
print("Count validation:", db_count == len(exceptions_db))

Python records: 685
PostgreSQL records: 685
Count validation: True


In [10]:
# Verify severity distribution

severity_query = """
SELECT
    severity,
    COUNT(*) AS total_exceptions
FROM exception_records
GROUP BY severity
ORDER BY severity;
"""

with engine.connect() as connection:
    severity_results = pd.read_sql(
        text(severity_query),
        connection
    )

display(severity_results)

,severity,total_exceptions
0,HIGH,185
1,MEDIUM,500


In [11]:
status_query = """
SELECT
    status,
    COUNT(*) AS total_exceptions
FROM exception_records
GROUP BY status
ORDER BY status;
"""

with engine.connect() as connection:
    status_results = pd.read_sql(
        text(status_query),
        connection
    )

display(status_results)

,status,total_exceptions
0,OPEN,684
1,RESOLVED,1


In [13]:
relationship_query = """
SELECT
    COUNT(*) AS total_exceptions,
    COUNT(DISTINCT e.bag_id) AS unique_bags,
    COUNT(*) FILTER (
        WHERE b.bag_id IS NULL
    ) AS invalid_baggage_references
FROM exception_records e
LEFT JOIN baggage b
    ON e.bag_id = b.bag_id;
"""

with engine.connect() as connection:
    relationship_results = pd.read_sql(
        text(relationship_query),
        connection
    )

display(relationship_results)

,total_exceptions,unique_bags,invalid_baggage_references
0,685,500,0


In [14]:
null_query = """
SELECT
    COUNT(*) FILTER (WHERE assigned_to IS NULL) AS unassigned_exceptions,
    COUNT(*) FILTER (WHERE resolved_at IS NULL) AS unresolved_exceptions,
    COUNT(*) FILTER (WHERE resolution_notes IS NULL) AS missing_resolution_notes
FROM exception_records;
"""

with engine.connect() as connection:
    null_results = pd.read_sql(text(null_query), connection)

display(null_results)

,unassigned_exceptions,unresolved_exceptions,missing_resolution_notes
0,684,684,684


In [15]:
final_validation_query = """
SELECT
    COUNT(*) AS total_exceptions,
    COUNT(DISTINCT exception_id) AS unique_exception_ids,
    COUNT(DISTINCT bag_id) AS unique_bags,
    COUNT(*) FILTER (WHERE severity = 'HIGH') AS high_exceptions,
    COUNT(*) FILTER (WHERE severity = 'MEDIUM') AS medium_exceptions,
    COUNT(*) FILTER (WHERE status = 'OPEN') AS open_exceptions,
    COUNT(*) FILTER (WHERE status = 'RESOLVED') AS resolved_exceptions
FROM exception_records;
"""

with engine.connect() as connection:
    final_validation_results = pd.read_sql(
        text(final_validation_query),
        connection
    )

display(final_validation_results)

,total_exceptions,unique_exception_ids,unique_bags,high_exceptions,medium_exceptions,open_exceptions,resolved_exceptions
0,685,685,500,185,500,684,1


In [1]:
from pathlib import Path
import sys

project_path = Path.cwd()
processed_path = project_path / "data" / "processed"

print("Python Environment:", sys.executable)
print("Project Path:", project_path)
print("Processed Folder Exists:", processed_path.exists())

Python Environment: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\.venv\Scripts\python.exe
Project Path: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets
Processed Folder Exists: True


In [2]:
import pandas as pd

files = [
    "phase13_journey_analytics.csv",
    "phase13_exception_analytics.csv",
    "phase13_airport_analytics.csv",
    "phase13_flight_analytics.csv",
    "phase13_connection_analytics.csv",
    "phase13_dwell_time_analytics.csv"
]

for filename in files:
    file_path = processed_path / filename

    if file_path.exists():
        df = pd.read_csv(file_path)
        print(
            filename,
            "| Rows:", len(df),
            "| Columns:", len(df.columns)
        )
    else:
        print(filename, "| FILE NOT FOUND")

phase13_journey_analytics.csv | Rows: 2 | Columns: 5
phase13_exception_analytics.csv | Rows: 3 | Columns: 5
phase13_airport_analytics.csv | Rows: 10 | Columns: 5
phase13_flight_analytics.csv | Rows: 179 | Columns: 7
phase13_connection_analytics.csv | Rows: 500 | Columns: 12
phase13_dwell_time_analytics.csv | Rows: 3 | Columns: 5


In [3]:
for filename in files:
    file_path = processed_path / filename
    df = pd.read_csv(file_path)

    print("\n" + "=" * 65)
    print("DATASET:", filename)
    print("=" * 65)

    print("\nCOLUMN NAMES:")
    print(df.columns.tolist())

    print("\nDATA TYPES:")
    print(df.dtypes)

    print("\nSAMPLE RECORDS:")
    display(df.head(3))


DATASET: phase13_journey_analytics.csv

COLUMN NAMES:
['journey_type', 'total_baggage', 'baggage_with_issues', 'investigation_rate', 'percentage_of_total']

DATA TYPES:
journey_type               str
total_baggage            int64
baggage_with_issues      int64
investigation_rate     float64
percentage_of_total    float64
dtype: object

SAMPLE RECORDS:


,journey_type,total_baggage,baggage_with_issues,investigation_rate,percentage_of_total
0,CONNECTING,500,500,100.0,50.0
1,DIRECT,500,0,0.0,50.0



DATASET: phase13_exception_analytics.csv

COLUMN NAMES:
['exception_type', 'severity', 'status', 'total_exceptions', 'unique_baggage']

DATA TYPES:
exception_type        str
severity              str
status                str
total_exceptions    int64
unique_baggage      int64
dtype: object

SAMPLE RECORDS:


,exception_type,severity,status,total_exceptions,unique_baggage
0,HANDOFF_TIMING_INCONSISTENCY,HIGH,OPEN,184,184
1,HANDOFF_TIMING_INCONSISTENCY,HIGH,RESOLVED,1,1
2,MISSING_SEGMENT_EVENT,MEDIUM,OPEN,500,500



DATASET: phase13_airport_analytics.csv

COLUMN NAMES:
['airport_code', 'total_scan_events', 'unique_baggage', 'event_types', 'percentage_of_total_scans']

DATA TYPES:
airport_code                     str
total_scan_events              int64
unique_baggage                 int64
event_types                    int64
percentage_of_total_scans    float64
dtype: object

SAMPLE RECORDS:


,airport_code,total_scan_events,unique_baggage,event_types,percentage_of_total_scans
0,MAA,1676,419,4,18.622222
1,LHR,1317,439,3,14.633333
2,DXB,1088,290,8,12.088889


... (2 outputs left out)


DATASET: phase13_connection_analytics.csv

COLUMN NAMES:
['bag_id', 'missing_segment_event_count', 'missing_segment_events', 'unload_to_transfer_minutes', 'transfer_to_load_minutes', 'handoff_chronology_valid', 'handoff_issue', 'airport_consistent', 'missing_segment_event_detected', 'handoff_timing_inconsistency_detected', 'advanced_detection_status', 'advanced_severity']

DATA TYPES:
bag_id                                       str
missing_segment_event_count                int64
missing_segment_events                       str
unload_to_transfer_minutes               float64
transfer_to_load_minutes                 float64
handoff_chronology_valid                    bool
handoff_issue                                str
airport_consistent                          bool
missing_segment_event_detected              bool
handoff_timing_inconsistency_detected       bool
advanced_detection_status                    str
advanced_severity                            str
dtype: object

SAMPLE R

,bag_id,missing_segment_event_count,missing_segment_events,unload_to_transfer_minutes,transfer_to_load_minutes,handoff_chronology_valid,handoff_issue,airport_consistent,missing_segment_event_detected,handoff_timing_inconsistency_detected,advanced_detection_status,advanced_severity
0,BAG00501,1,SORTED,30.0,-810.0,False,TRANSFER_LOADING_TIME_INCONSISTENCY,True,True,True,MISSING_EVENT_AND_HANDOFF_TIMING_ISSUE,HIGH
1,BAG00502,1,SORTED,30.0,270.0,True,NO_TIMING_ISSUE,True,True,False,MISSING_EVENT_ONLY,MEDIUM
2,BAG00503,1,SORTED,30.0,270.0,True,NO_TIMING_ISSUE,True,True,False,MISSING_EVENT_ONLY,MEDIUM



DATASET: phase13_dwell_time_analytics.csv

COLUMN NAMES:
['transition', 'average_minutes', 'median_minutes', 'minimum_minutes', 'maximum_minutes']

DATA TYPES:
transition             str
average_minutes    float64
median_minutes     float64
minimum_minutes    float64
maximum_minutes    float64
dtype: object

SAMPLE RECORDS:


,transition,average_minutes,median_minutes,minimum_minutes,maximum_minutes
0,ARRIVED_TO_UNLOADED,15.0,15.0,15.0,15.0
1,UNLOADED_TO_TRANSFER,30.0,30.0,30.0,30.0
2,TRANSFER_TO_SECOND_LOAD,-129.6,270.0,-810.0,270.0


In [4]:
for filename in files:
    df = pd.read_csv(processed_path / filename)

    print("\n" + "=" * 65)
    print("DATASET:", filename)
    print("=" * 65)

    print("\nMISSING VALUES:")
    print(df.isnull().sum())

    print("\nDUPLICATE ROWS:", df.duplicated().sum())

    print("\nUNIQUE VALUES IN CATEGORICAL COLUMNS:")
    for column in df.select_dtypes(include=["object", "str"]).columns:
        print(f"\n{column}:")
        print(df[column].dropna().unique()[:20])


DATASET: phase13_journey_analytics.csv

MISSING VALUES:
journey_type           0
total_baggage          0
baggage_with_issues    0
investigation_rate     0
percentage_of_total    0
dtype: int64

DUPLICATE ROWS: 0

UNIQUE VALUES IN CATEGORICAL COLUMNS:

journey_type:
<StringArray>
['CONNECTING', 'DIRECT']
Length: 2, dtype: str

DATASET: phase13_exception_analytics.csv

MISSING VALUES:
exception_type      0
severity            0
status              0
total_exceptions    0
unique_baggage      0
dtype: int64

DUPLICATE ROWS: 0

UNIQUE VALUES IN CATEGORICAL COLUMNS:

exception_type:
<StringArray>
['HANDOFF_TIMING_INCONSISTENCY', 'MISSING_SEGMENT_EVENT']
Length: 2, dtype: str

severity:
<StringArray>
['HIGH', 'MEDIUM']
Length: 2, dtype: str

status:
<StringArray>
['OPEN', 'RESOLVED']
Length: 2, dtype: str

DATASET: phase13_airport_analytics.csv

MISSING VALUES:
airport_code                 0
total_scan_events            0
unique_baggage               0
event_types                  0
percent

In [5]:
try:
    import snowflake.connector
    print("Snowflake connector is installed.")
    print("Version:", snowflake.connector.__version__)
except ImportError:
    print("Snowflake connector is NOT installed.")

Snowflake connector is installed.
Version: 5.2.0


In [6]:
import snowflake.connector
from getpass import getpass

account = input("Enter Snowflake account identifier: ").strip()
user = input("Enter Snowflake username: ").strip()
password = getpass("Enter Snowflake password: ")

conn = snowflake.connector.connect(
    account=account,
    user=user,
    password=password,
    warehouse="AIRPORT_BAGGAGE_WH",
    database="AIRPORT_BAGGAGE_DB",
    schema="BRONZE_SCHEMA"
)

print("Connected to Snowflake successfully!")

Enter Snowflake account identifier:  XJXSNLZ-EL65065
Enter Snowflake username:  pranaviselvaraju@gmail.com
Enter Snowflake password:  ········


C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\.venv\Lib\site-packages\snowflake\connector\connection\_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


Connected to Snowflake successfully!


In [7]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        CURRENT_ACCOUNT(),
        CURRENT_USER(),
        CURRENT_DATABASE(),
        CURRENT_SCHEMA(),
        CURRENT_WAREHOUSE()
""")

result = cursor.fetchone()

print("Account:", result[0])
print("User:", result[1])
print("Database:", result[2])
print("Schema:", result[3])
print("Warehouse:", result[4])

Account: JQ44300
User: pranaviselvaraju
Database: AIRPORT_BAGGAGE_DB
Schema: BRONZE_SCHEMA
Warehouse: AIRPORT_BAGGAGE_WH


In [9]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "pyarrow"
])

0

In [10]:
import pyarrow
print("PyArrow version:", pyarrow.__version__)

PyArrow version: 26.0.1


In [11]:
from snowflake.connector.pandas_tools import write_pandas

df = pd.read_csv(processed_path / "phase13_journey_analytics.csv")
df.columns = [column.upper() for column in df.columns]

cursor.execute("""
    SELECT COUNT(*)
    FROM AIRPORT_BAGGAGE_DB.BRONZE_SCHEMA.BRONZE_JOURNEY_ANALYTICS
""")

existing_count = cursor.fetchone()[0]

if existing_count == 0:
    success, nchunks, nrows, output = write_pandas(
        conn,
        df,
        table_name="BRONZE_JOURNEY_ANALYTICS",
        database="AIRPORT_BAGGAGE_DB",
        schema="BRONZE_SCHEMA",
        auto_create_table=False,
        overwrite=False
    )

    print("Upload successful:", success)
    print("Rows uploaded:", nrows)
else:
    print("Table already contains", existing_count, "rows.")
    print("Skipping upload to prevent duplicates.")

Upload successful: True
Rows uploaded: 2


In [12]:
cursor.execute("""
    SELECT *
    FROM AIRPORT_BAGGAGE_DB.BRONZE_SCHEMA.BRONZE_JOURNEY_ANALYTICS
    ORDER BY JOURNEY_TYPE
""")

journey_records = cursor.fetchall()

for record in journey_records:
    print(record)

('CONNECTING', 500, 500, 100.0, 50.0)
('DIRECT', 500, 0, 0.0, 50.0)


In [13]:
datasets = {
    "phase13_exception_analytics.csv": "BRONZE_EXCEPTION_ANALYTICS",
    "phase13_airport_analytics.csv": "BRONZE_AIRPORT_ANALYTICS",
    "phase13_flight_analytics.csv": "BRONZE_FLIGHT_ANALYTICS",
    "phase13_connection_analytics.csv": "BRONZE_CONNECTION_ANALYTICS",
    "phase13_dwell_time_analytics.csv": "BRONZE_DWELL_TIME_ANALYTICS"
}

for filename, table_name in datasets.items():

    df = pd.read_csv(processed_path / filename)
    df.columns = [column.upper() for column in df.columns]

    cursor.execute(f"""
        SELECT COUNT(*)
        FROM AIRPORT_BAGGAGE_DB.BRONZE_SCHEMA.{table_name}
    """)

    existing_count = cursor.fetchone()[0]

    if existing_count == 0:
        success, nchunks, nrows, output = write_pandas(
            conn,
            df,
            table_name=table_name,
            database="AIRPORT_BAGGAGE_DB",
            schema="BRONZE_SCHEMA",
            auto_create_table=False,
            overwrite=False
        )

        print(
            f"{table_name} | Success: {success} | "
            f"Rows uploaded: {nrows}"
        )
    else:
        print(
            f"{table_name} | Already contains "
            f"{existing_count} rows. Skipped."
        )

BRONZE_EXCEPTION_ANALYTICS | Success: True | Rows uploaded: 3
BRONZE_AIRPORT_ANALYTICS | Success: True | Rows uploaded: 10
BRONZE_FLIGHT_ANALYTICS | Success: True | Rows uploaded: 179
BRONZE_CONNECTION_ANALYTICS | Success: True | Rows uploaded: 500
BRONZE_DWELL_TIME_ANALYTICS | Success: True | Rows uploaded: 3


In [14]:
expected_counts = {
    "BRONZE_JOURNEY_ANALYTICS": 2,
    "BRONZE_EXCEPTION_ANALYTICS": 3,
    "BRONZE_AIRPORT_ANALYTICS": 10,
    "BRONZE_FLIGHT_ANALYTICS": 179,
    "BRONZE_CONNECTION_ANALYTICS": 500,
    "BRONZE_DWELL_TIME_ANALYTICS": 3
}

total_rows = 0

for table_name, expected in expected_counts.items():

    cursor.execute(f"""
        SELECT COUNT(*)
        FROM AIRPORT_BAGGAGE_DB.BRONZE_SCHEMA.{table_name}
    """)

    actual = cursor.fetchone()[0]
    total_rows += actual

    status = "PASS" if actual == expected else "CHECK"

    print(
        f"{table_name}: Expected = {expected}, "
        f"Actual = {actual}, Status = {status}"
    )

print("\nTotal rows in Bronze:", total_rows)

BRONZE_JOURNEY_ANALYTICS: Expected = 2, Actual = 2, Status = PASS
BRONZE_EXCEPTION_ANALYTICS: Expected = 3, Actual = 3, Status = PASS
BRONZE_AIRPORT_ANALYTICS: Expected = 10, Actual = 10, Status = PASS
BRONZE_FLIGHT_ANALYTICS: Expected = 179, Actual = 179, Status = PASS
BRONZE_CONNECTION_ANALYTICS: Expected = 500, Actual = 500, Status = PASS
BRONZE_DWELL_TIME_ANALYTICS: Expected = 3, Actual = 3, Status = PASS

Total rows in Bronze: 697
